# Advanced IR Project Part 2

Version 1.1

In the Project Part 2 assessment item of Advanced IR you will build a retrieval pipeline for web documents.

#### Grading

This project is assessed using the following criteria (20pts in total):

Implementation (10pts)
 - Code quality and comments (2.5pts)
 - Comparison to a BM25 baseline and your own model from Part 1 (2.5pts)
 - Solution thoughtfully designed to tackle the specific search problem  (consideration of query and document properties, choice of model architectures, etc.) (5pts)

Discussion (10pts)
 - Presentation, spelling, and grammar (5pts)
 - Appropriate and thoughtful responses to prompts (5pts)

Bonus

The top three submissions (not systems) to the leaderboard (see below) will be awarded bonus points:
 - 1st place (5pts)
 - 2nd place (2.5pts)
 - 3rd place (2.5pts)

#### Submission

Export the notebook as a PDF document with all of the code cells executed to show the outputs and the discussion filled out. Submit the PDF of this notebook to ILIAS ([project link](https://ovidius.uni-tuebingen.de/ilias3/ilias.php?baseClass=ilexercisehandlergui&cmdNode=cp:o6&cmdClass=ilObjExerciseGUI&cmd=showOverview&ref_id=5379330)) by the 13.01.2026 23:55.

If you have developed any code outside of the notebook for the experiments, please include this additional code in a ZIP file alongside the original and PDF versions of this notebook. 

#### Leaderboard

Project Part 2 also has a leaderboard component to see how your search system compares to everyone else in the class. You can find the leaderboard here: https://ir-leaderboard.scells.me/

To add a run to the leaderboard, visit https://ir-leaderboard.scells.me/upload?secret=YOUR_MATRICULATION_NUMBER You might want to go back to the first tutorial which explains more about this run format. You may also run `trec_eval` locally to validate that your run will be accepted by the online leaderboard.

Your name (as per ILIAS) will appear as the submitter, and the `runid` column in your run file (the last column) will appear as the system name. You can submit multiple systems and they will appear as separate rows in the leaderboard. Some decent baselines are already provided to try and beat. If you would prefer to use a pseudonym, please contact me by email (harrisen.scells@uni-tuebingen.de) and I will make the change for you; but please know that submitter names and system names cannot be changed once added to the leaderboard.

#### Tips

The test collection used in this lab is only available from [bwJupyter](https://hub.bwjupyter.de/services/profilemanagement/add?profile=061a82a3-a416-443b-86ec-c396129acdd3). You likely will not have enough free credits to run your experiments on Google Colab, and bwJupyter provides much better GPUs than Colab that will make your experiments much faster to run.

In this project, you should try to be imaginative and go beyond what has been shown in the tutorial. You are expected to read the pyterrier documentation to learn how to do things (like [understanding the operators](https://pyterrier.readthedocs.io/en/latest/operators.html), [using neural rankers](https://pyterrier.readthedocs.io/en/latest/neural.html), [document processing](https://pyterrier.readthedocs.io/en/latest/text.html#working-with-passages-rather-than-documents), and the many other features available). You are of course not limited to pyterrier if you know about other tools you would like to use, like, e.g., [pyserini](https://github.com/castorini/pyserini). 

If you are interested in running LLMs, huggingface has good [tutorials](https://huggingface.co/docs/transformers/en/llm_tutorial) and the GWDG can provide you with free [API keys](https://docs.hpc.gwdg.de/services/saia/index.html).

Furthermore, this project involves messy documents from webpages. These documents are much longer than the ones you've dealt with so far and will require some sort of processing to be able to use neural ranking models.

In [1]:
# These libraries are required to be able to access the dataset in pyterrier.
!pip install ir-datasets pyterrier git+https://github.com/tira-io/tira@d914806859f487037e7193a0773895937f39e8de#subdirectory=python-client

# Use this cell to install your other dependencies.
!pip install torch lightning-ir sentence_transformers transformers pyterrier-alpha openai codecarbon

  Cloning https://github.com/tira-io/tira (to revision d914806859f487037e7193a0773895937f39e8de) to /tmp/pip-req-build-r86pcm9h
  Running command git clone --filter=blob:none --quiet https://github.com/tira-io/tira /tmp/pip-req-build-r86pcm9h
  Running command git rev-parse -q --verify 'sha^d914806859f487037e7193a0773895937f39e8de'
  Running command git fetch -q https://github.com/tira-io/tira d914806859f487037e7193a0773895937f39e8de
  Running command git checkout -q d914806859f487037e7193a0773895937f39e8de
  Resolved https://github.com/tira-io/tira to commit d914806859f487037e7193a0773895937f39e8de
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [26]:
# # Install OpenJDK from conda-forge
# !conda install -c conda-forge openjdk -y

In [2]:
from tira.ir_datasets_util import register_dataset
from pyterrier.datasets import get_dataset

# On the bwJupyter cluster, this file exists in this shared path.
# Please, do not publish this data openly if you must download it.
register_dataset("__shared/clueweb12.zip", "clueweb12-subsample")
dataset = get_dataset("irds:clueweb12-subsample")

## Implementation

Use as many cells as you like below to implement your model on the dataset provided above. Once you've implemented your model compare it to BM25 and the pipeline you created in project Part 1. You should use the evaluation measures listed on the leaderboard system to evaluate your systems.

In [3]:
# Your solution here

In [4]:
import os
import datetime
import torch
import pandas as pd
import pyterrier as pt
from pathlib import Path
from pyterrier.measures import nDCG, P, RR
from ir_measures import nDCG, RR, P

from lightning_ir_pyterrier import BiEncoder, CrossEncoder
from lightning_ir import TorchDenseIndexConfig, TorchDenseSearchConfig
from pyterrier.text import get_text
import pyterrier_alpha as pta


device = "cpu"
if torch.cuda.is_available():
    device = "cuda"
if torch.backends.mps.is_available():
    device = "mps"

In [4]:
for doc in dataset.get_corpus_iter():
    # print("Raw Object:", doc)
    # Get just the field names
    if hasattr(doc, '_fields'):
        print("Fields available:", doc._fields)
    else:
        # Fallback if it's a standard dictionary
        print("Fields available:", doc.keys())
    break

clueweb12-subsample documents:   0%|          | 0/84241 [00:00<?, ?it/s]

Fields available: dict_keys(['text', 'docno'])


### Comparison to a BM25 baseline and your own model from Part 1

In [5]:
if not pt.java.started():
    pt.init()
    
# Updated Cascade Model from Part 1:
K_CANDIDATES = 1000       # BM25 retrieves 1000

# Indexer
pt_index_path = "./index_clueweb12-subsample"
if not os.path.exists(pt_index_path + "/data.properties"):
  indexer = pt.index.IterDictIndexer(
      pt_index_path, 
      meta_reverse=[],
      fields=["text"],
      meta={'docno': 25}
  )
  index_ref = indexer.index(dataset.get_corpus_iter())
else:
  index_ref = pt_index_path
index = pt.IndexFactory.of(index_ref)

# Baseline Model: truncated to the same cut-off of K_CANDIDATES for better evaluation analysis
bm25_baseline = (pt.terrier.Retriever(index, wmodel="BM25") % K_CANDIDATES)

# Simple Cross Encoder
cross_encoder = CrossEncoder(
    model_name_or_path="webis/monoelectra-base", 
    batch_size=1,
    verbose=True,
    use_gpu=torch.cuda.is_available()
)
cross_encoder._module.to(device)
cross_encoder._module.device

# Cascade Pipeline (BM25 >> cross_encoder)
# Stage 1: BM25 retrieves K_CANDIDATES candidates
bm25_stage1 = bm25_baseline

# Stage 2: Cross-Encoder re-ranks the K_CANDIDATES
cascade_model = (
    bm25_stage1
    >> get_text(dataset) 
    >> cross_encoder
)

Java started and loaded: pyterrier.java.colab, pyterrier.java, pyterrier.java.24, pyterrier.terrier.java [version=5.11 (build: craig.macdonald 2025-01-13 21:29), helper_version=0.0.8]
/tmp/ipykernel_3435548/3334950383.py:2: DeprecationWarning: Call to deprecated method pt.init(). Deprecated since version 0.11.0.
java is now started automatically with default settings. To force initialisation early, run:
pt.java.init() # optional, forces java initialisation
  pt.init()
You are using a model of type mono to instantiate a model of type electra. This is not supported for all configurations of models and can yield errors.
You are using a model of type mono to instantiate a model of type electra. This is not supported for all configurations of models and can yield errors.
You are using a model of type mono to instantiate a model of type electra. This is not supported for all configurations of models and can yield errors.
You are using a model of type mono to instantiate a model of type elect

In [12]:
# nDCG@k: normalised discounted cumulative gain
# P@k: precision at depth k, fraction of top-k are relevant
# RR: Rank of the first relevant item
topics = dataset.get_topics(variant='text')
print(f"Running experiment on {len(topics)} topics, evaluating at k={K_CANDIDATES}...")
experiment_results = pt.Experiment(
    [
        bm25_baseline,
        cascade_model
    ],
    dataset.get_topics(variant='text'),  
    dataset.get_qrels(),   
    eval_metrics=[nDCG@10, nDCG@100, RR, P@10],
    names=["BM25 Baseline", "Cascade Model from Part 1 (BM25 >> Cross-Encoder)"],
    verbose=True,
)
print(experiment_results)

Running experiment on 50 topics, evaluating at k=1000...


webis/monoelectra-base rank: 0batch [00:00, ?batch/s]

/home/kpch/miniconda3/envs/py312/lib/python3.12/site-packages/pyterrier/_evaluation/_validation.py:43: UserWarning: Transformer Cascade Model from Part 1 (BM25 >> Cross-Encoder) ((TerrierRetr(BM25) >> RankCutoff(1000) >> <pyterrier.datasets._irds.IRDSTextLoader object at 0x7e0a695caed0> >> <lightning_ir_pyterrier.CrossEncoder object at 0x7e0a695cb800>)) at position 1 failed to validate: Cannot determine outputs for (TerrierRetr(BM25) >> RankCutoff(1000) >> <pyterrier.datasets._irds.IRDSTextLoader object at 0x7e0a695caed0> >> <lightning_ir_pyterrier.CrossEncoder object at 0x7e0a695cb800>) with inputs: ['qid', 'query'] - if your pipeline works, set validate='ignore' to remove this warning, or add transform_output method to the transformers in this pipeline to clarify how it works. See https://pyterrier.readthedocs.io/en/latest/troubleshooting/inspection.html for more information.
  warn(


pt.Experiment:   0%|          | 0/2 [00:00<?, ?system/s]

webis/monoelectra-base rank:   0%|          | 0/50000 [00:00<?, ?batch/s]

                                                name        RR   P@10  \
0                                      BM25 Baseline  0.550403  0.504   
1  Cascade Model from Part 1 (BM25 >> Cross-Encoder)  0.587185  0.350   

    nDCG@10  nDCG@100  
0  0.306496  0.362850  
1  0.254421  0.294331  


### Retrieval pipeline for web documents

In [11]:
# Calculate lengths for all documents in the corpus
# We split by whitespace to count words
doc_lengths = [len(doc['text'].split()) for doc in dataset.get_corpus_iter()]

if doc_lengths:
    avg_len = sum(doc_lengths) / len(doc_lengths)
    max_len = max(doc_lengths)
    min_len = min(doc_lengths)
else:
    avg_len = 0
    max_len = 0
    min_len = 0

print(f"Average Document Length: {avg_len:.2f} words")
print(f"Maximum Document Length: {max_len} words")
print(f"Minimum Document Length: {min_len} words")

clueweb12-subsample documents:   0%|          | 0/84241 [00:00<?, ?it/s]

Average Document Length: 2110.74 words
Maximum Document Length: 335925 words
Minimum Document Length: 0 words


In [6]:
# Initialize PyTerrier (Java Check)
if not pt.java.started():
    pt.init()

In [26]:
# Check if we have raw HTML tags
for doc in dataset.get_corpus_iter():
    print(doc["text"][:500])
    break

clueweb12-subsample documents:   0%|          | 0/84241 [00:00<?, ?it/s]

Louis Vuitton Damier Ebene Canvas Belts Stylish-0004,Cheap Louis Vuitton Belts,wholesale Louis Vuitton Leather Belts Home Log In
Currencies:
Languages: English
Shopping Cart: now in your cart 0 items
 
Home
  • Return & Exchange
  • Shipping
  • Payment
  • About wholesale
  • Contact Us
  • Live chat
    • Louis Vuitton Bags
      Louis Vuitton 2010 New Style Louis Vuitton Damier Handbags Louis Vuitton Epi Leather Louis Vuitton Leather Bags Louis Vuitton Men Handbags Louis Vuitton Monogram Hand


In [7]:
# BUILD INDEXES 
def build_indexes(dataset, sparse_path, dense_path, bi_model_id, batch_size=32):
    # A. Sparse
    if not os.path.exists(sparse_path + "/data.properties"):
        print(f"--- Building Sparse Index at {sparse_path} ---")
        indexer = pt.IterDictIndexer(
            sparse_path, 
            meta_reverse=['docno'],                     # Allows looking up docid integers from string docnos
            meta={'docno': 25, 'text': 32000},          # Store up to ~32k characters so the footer is actually saved
            blocks=True,                                # Enables phrase search ("query terms")
            fields=['text'] 
        )
        indexer.index(dataset.get_corpus_iter())

    # B. Dense (Bi-Encoder)
    safe_model_name = bi_model_id.replace("/", "_")
    specific_dense_path = os.path.join(dense_path, safe_model_name)
    
    if not os.path.exists(specific_dense_path):
        print(f"--- Building Dense Index at {specific_dense_path} ---")
        bi_indexer = BiEncoder(
            model_name_or_path=bi_model_id,
            index_dir=specific_dense_path,
            index_config=TorchDenseIndexConfig(),
            text_field="text",
            batch_size=batch_size,
            use_gpu=True,
            verbose=True
        )
        if hasattr(bi_indexer, "_module"):
             bi_indexer._module.to("cuda" if torch.cuda.is_available() else "cpu")
        bi_indexer.index(dataset.get_corpus_iter())
        
    return specific_dense_path

In [8]:
def run_toggled_experiment(
    dataset, 
    sparse_path, 
    dense_index_path,
    topics_variant='text',
    
    # --- TOGGLES ---
    use_bm25f=False,        # Use Title weights?
    use_rm3=False,          # Use Query Expansion?
    use_bi_encoder=False,   # Use Dense Retrieval?
    use_rrf=False,          # Use Rank Fusion?
    use_cross_encoder=True, # Use Final Reranker?

    # --- PARAMETERS ---
    bi_model_id="webis/bert-bi-encoder",
    ce_model_id="webis/monoelectra-base",
    
    # if using bm25f, but out docs don't have a predefined 'title' and extracting doesn't seem worth the effort
    bm25f_params={"w.title": 3.0, "w.text": 1.0}, 
    
    rm3_terms=10,
    rm3_docs=10,
    window_len=300,        # 300 tokens is robust for web docs
    window_stride=150,     # 50% overlap
    bi_top_k=1000,         # Rerank depth for Bi-Encoder
    ce_top_k=100,          # Rerank depth for Cross-Encoder
    
    batch_size=32
):
    
    index = pt.IndexFactory.of(sparse_path)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    
    # --- STAGE 1: Sparse Retrieval ---
    if use_bm25f:
        print(" [1] Using BM25F (Fielded)")
        controls = {"wmodel": "BM25F"}
        controls.update(bm25f_params)
        pipeline = pt.terrier.Retriever(index, controls=controls)
        name = "BM25F"
    else:
        print(" [1] Using Standard BM25")
        pipeline = pt.terrier.Retriever(index, wmodel="BM25")
        name = "BM25"

    # --- STAGE 2: Query Expansion ---
    if use_rm3:
        print(f" [2] Adding RM3 ({rm3_terms} terms)")
        rm3 = pt.rewrite.RM3(index, fb_terms=rm3_terms, fb_docs=rm3_docs)
        pipeline = pipeline >> rm3 >> pipeline
        name += "+RM3"
    
    # --- STAGE 3: Dense Retrieval (Bi-Encoder) ---
    if use_bi_encoder:
        print(f" [3] Adding Bi-Encoder ({bi_model_id})")
        
        # 1. Define the Bi-Encoder
        bi_enc = BiEncoder(
            model_name_or_path=bi_model_id,
            index_dir=Path(dense_index_path),  # It needs the index to search!
            index_config=TorchDenseIndexConfig(),
            search_config=TorchDenseSearchConfig(k=bi_top_k),
            text_field="text",
            batch_size=batch_size,
            use_gpu=True
        )
        if hasattr(bi_enc, "_module"): bi_enc._module.to(device)

        # 2. Define the Branch (PARALLEL)
        # We do NOT put 'pipeline >>' in front. 
        # We treat 'bi_enc' as a standalone retriever that starts from the user query.
        dense_pipe = bi_enc
        
        # 3. Apply Fusion
        if use_rrf:
            print("     -> Applying Fusion (RRF)")
            # pipeline = BM25 results
            # dense_pipe = Bi-Encoder results (New docs!)
            # RRFusion merges them together
            pipeline = pta.fusion.RRFusion(pipeline, dense_pipe, k=60)
            name += "+BiEnc+RRF"
        else:
            # If RRF is off, we strictly replace BM25 with Bi-Encoder
            pipeline = dense_pipe
            name = "BiEnc"

    # --- STAGE 4: Cross-Encoder ---
    if use_cross_encoder:
        print(f" [4] Adding Cross-Encoder ({ce_model_id})")
        ce = CrossEncoder(
            model_name_or_path=ce_model_id,
            text_field="text",
            batch_size=batch_size, 
            use_gpu=True
        )
        if hasattr(ce, "_module"): ce._module.to(device)

        pipeline = (
            (pipeline % ce_top_k) 
            >> pt.text.get_text(dataset, "text") 
            >> pt.text.sliding(length=window_len, stride=window_stride, text_attr='text', prepend_attr=None) # otherwise looks for "body"
            >> ce 
            >> pt.text.max_passage()
        )
        name += "+CE"

    # --- Execution ---
    print(f"\nRunning Pipeline: {name}")
    
    # We run .transform() first. This does the retrieval + cross-encoding
    start_time = datetime.datetime.now()
    run_out = pipeline.transform(dataset.get_topics(variant=topics_variant))
    print(f"Pipeline finished in {datetime.datetime.now() - start_time}")

    # Save results
    timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M")
    safe_name = name.replace("+", "_").replace("(", "").replace(")", "")
    pt.io.write_results(run_out, f"run_{timestamp}_{safe_name}.res")

    # Calculate Metrices
    metrics = [nDCG@10, nDCG@100, P@10, RR(rel=1)]
    exp = pt.Experiment(
        [run_out],  # <--- Pass the result dataframe, not the pipeline: PyTerrier sees it's already data and just computes the math
        dataset.get_topics(variant=topics_variant),
        dataset.get_qrels(),
        eval_metrics=metrics,
        names=[name],
        verbose=True,        # shows progress
        validate=False       # surpress warnings
    )
    exp.to_csv(f"results_{timestamp}_{safe_name}.csv")
    
    print(exp)
    return exp

In [9]:
# MAIN
SPARSE_PATH = "./indices/sparse_clueweb"
DENSE_ROOT  = "./indices/dense_clueweb"
BI_MODEL = "webis/bert-bi-encoder" 
CE_MODEL = "webis/monoelectra-large"
    
# Step 1. Build Indexes
dense_path = build_indexes(dataset, SPARSE_PATH, DENSE_ROOT, BI_MODEL)

In [30]:
# Step 2. Run Experiment
run_toggled_experiment(
    dataset,
    SPARSE_PATH,
    dense_path,
    topics_variant='text',
        
    # --- TOGGLES ---
    use_bm25f=False,       # Strong title baseline
    use_rm3=False,         # Essential for short queries
    use_bi_encoder=True,   # Optional: Turn on for deep semantic search
    use_rrf=True,          # Optional: Only works if bi_encoder is True
    use_cross_encoder=True,# The precision booster
        
    # --- PARAMETERS ---
    bi_model_id=BI_MODEL,
    ce_model_id=CE_MODEL,
    # bm25f_params={"w.title": 5.0, "w.text": 1.0}, 
    # rm3_terms=10,
    # rm3_docs=10,
    window_len=300,        # 300 tokens is robust for web docs
    window_stride=150,     # 50% overlap
    bi_top_k=1000,         # Rerank depth for Bi-Encoder
    ce_top_k=100,          # Rerank depth for Cross-Encoder
)

 [1] Using Standard BM25
 [3] Adding Bi-Encoder (webis/bert-bi-encoder)


You are using a model of type lir-dpr to instantiate a model of type bert. This is not supported for all configurations of models and can yield errors.
You are using a model of type lir-dpr to instantiate a model of type bert. This is not supported for all configurations of models and can yield errors.
You are using a model of type lir-dpr to instantiate a model of type bert. This is not supported for all configurations of models and can yield errors.
You are using a model of type lir-dpr to instantiate a model of type bert. This is not supported for all configurations of models and can yield errors.
The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'BertTokenizer'. 
The class this function is called from is 'BiEncoderBertTokenizerFast'.


     -> Applying Fusion (RRF)
 [4] Adding Cross-Encoder (webis/monoelectra-large)


You are using a model of type mono to instantiate a model of type electra. This is not supported for all configurations of models and can yield errors.
You are using a model of type mono to instantiate a model of type electra. This is not supported for all configurations of models and can yield errors.
You are using a model of type mono to instantiate a model of type electra. This is not supported for all configurations of models and can yield errors.
You are using a model of type mono to instantiate a model of type electra. This is not supported for all configurations of models and can yield errors.
The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'ElectraTokenizer'. 
The class this function is called from is 'CrossEncoderElectraTokenizerFast'.



Running Pipeline: BM25+BiEnc+RRF+CE
calling sliding on df of 5000 rows


passaging:   0%|          | 0/5000 [00:00<?, ?it/s]

Pipeline finished in 0:03:39.212369


pt.Experiment:   0%|          | 0/1 [00:00<?, ?system/s]

                name        RR   P@10   nDCG@10  nDCG@100
0  BM25+BiEnc+RRF+CE  0.638627  0.428  0.315834  0.362112


,name,RR,P@10,nDCG@10,nDCG@100
0,BM25+BiEnc+RRF+CE,0.638627,0.428,0.315834,0.362112


In [111]:
run_toggled_experiment(
    dataset,
    SPARSE_PATH,
    dense_path,
    topics_variant='text',
        
    # --- TOGGLES  ---
    use_bm25f=False,       # Strong title baseline
    use_rm3=False,         # Essential for short queries
    use_bi_encoder=True,   # Optional: Turn on for deep semantic search
    use_rrf=True,          # Optional: Only works if bi_encoder is True
    use_cross_encoder=True,# The precision booster
        
    # --- PARAMETERS ---
    bi_model_id=BI_MODEL,
    ce_model_id="webis/monoelectra-base",
    # bm25f_params={"w.title": 5.0, "w.text": 1.0}, 
    # rm3_terms=10,
    # rm3_docs=10,
    window_len=300,        # 300 tokens is robust for web docs
    window_stride=150,     # 50% overlap
    bi_top_k=1000,         # Rerank depth for Bi-Encoder
    ce_top_k=100,          # Rerank depth for Cross-Encoder
)

 [1] Using Standard BM25
 [3] Adding Bi-Encoder (webis/bert-bi-encoder)


You are using a model of type lir-dpr to instantiate a model of type bert. This is not supported for all configurations of models and can yield errors.
You are using a model of type lir-dpr to instantiate a model of type bert. This is not supported for all configurations of models and can yield errors.
You are using a model of type lir-dpr to instantiate a model of type bert. This is not supported for all configurations of models and can yield errors.
You are using a model of type lir-dpr to instantiate a model of type bert. This is not supported for all configurations of models and can yield errors.
The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'BertTokenizer'. 
The class this function is called from is 'BiEncoderBertTokenizerFast'.


     -> Applying Fusion (RRF)
 [4] Adding Cross-Encoder (webis/monoelectra-base)


You are using a model of type mono to instantiate a model of type electra. This is not supported for all configurations of models and can yield errors.
You are using a model of type mono to instantiate a model of type electra. This is not supported for all configurations of models and can yield errors.
You are using a model of type mono to instantiate a model of type electra. This is not supported for all configurations of models and can yield errors.
You are using a model of type mono to instantiate a model of type electra. This is not supported for all configurations of models and can yield errors.
The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'ElectraTokenizer'. 
The class this function is called from is 'CrossEncoderElectraTokenizerFast'.



Running Pipeline: BM25+BiEnc+RRF+CE
calling sliding on df of 5000 rows


passaging:   0%|          | 0/5000 [00:00<?, ?it/s]

Pipeline finished in 0:01:32.867664


pt.Experiment:   0%|          | 0/1 [00:00<?, ?system/s]

                name        RR   P@10   nDCG@10  nDCG@100
0  BM25+BiEnc+RRF+CE  0.624785  0.424  0.307404  0.359267


,name,RR,P@10,nDCG@10,nDCG@100
0,BM25+BiEnc+RRF+CE,0.624785,0.424,0.307404,0.359267


In [112]:
run_toggled_experiment(
    dataset,
    SPARSE_PATH,
    dense_path,
    topics_variant='text',
        
    # --- TOGGLES ---
    use_bm25f=False,       # Strong title baseline
    use_rm3=False,         # Essential for short queries
    use_bi_encoder=False,  # Optional: Turn on for deep semantic search
    use_rrf=False,         # Optional: Only works if bi_encoder is True
    use_cross_encoder=True,# The precision booster
        
    # --- PARAMETERS ---
    bi_model_id=BI_MODEL,
    ce_model_id=CE_MODEL,
    # bm25f_params={"w.title": 5.0, "w.text": 1.0}, 
    # rm3_terms=10,
    # rm3_docs=10,
    window_len=300,        # 300 tokens is robust for web docs
    window_stride=150,     # 50% overlap
    bi_top_k=1000,         # Rerank depth for Bi-Encoder
    ce_top_k=100,          # Rerank depth for Cross-Encoder
)

 [1] Using Standard BM25
 [4] Adding Cross-Encoder (webis/monoelectra-large)


You are using a model of type mono to instantiate a model of type electra. This is not supported for all configurations of models and can yield errors.
You are using a model of type mono to instantiate a model of type electra. This is not supported for all configurations of models and can yield errors.
You are using a model of type mono to instantiate a model of type electra. This is not supported for all configurations of models and can yield errors.
You are using a model of type mono to instantiate a model of type electra. This is not supported for all configurations of models and can yield errors.
The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'ElectraTokenizer'. 
The class this function is called from is 'CrossEncoderElectraTokenizerFast'.



Running Pipeline: BM25+CE
calling sliding on df of 5000 rows


passaging:   0%|          | 0/5000 [00:00<?, ?it/s]

Pipeline finished in 0:03:56.058426


pt.Experiment:   0%|          | 0/1 [00:00<?, ?system/s]

      name        RR   P@10   nDCG@10  nDCG@100
0  BM25+CE  0.718799  0.526  0.374639  0.390035


,name,RR,P@10,nDCG@10,nDCG@100
0,BM25+CE,0.718799,0.526,0.374639,0.390035


In [115]:
run_toggled_experiment(
    dataset,
    SPARSE_PATH,
    dense_path,
    topics_variant='text',
        
    # --- TOGGLES ---
    use_bm25f=False,       # Strong title baseline
    use_rm3=True,          # Essential for short queries
    use_bi_encoder=True,   # Optional: Turn on for deep semantic search
    use_rrf=True,          # Optional: Only works if bi_encoder is True
    use_cross_encoder=True,# The precision booster
        
    # --- PARAMETERS ---
    bi_model_id=BI_MODEL,
    ce_model_id=CE_MODEL,
    rm3_terms=12,
    rm3_docs=12,
    window_len=300,        # 300 tokens is robust for web docs
    window_stride=150,     # 50% overlap
    bi_top_k=1000,         # Rerank depth for Bi-Encoder
    ce_top_k=100,          # Rerank depth for Cross-Encoder
)

 [1] Using Standard BM25
 [2] Adding RM3 (12 terms)
 [3] Adding Bi-Encoder (webis/bert-bi-encoder)


You are using a model of type lir-dpr to instantiate a model of type bert. This is not supported for all configurations of models and can yield errors.
You are using a model of type lir-dpr to instantiate a model of type bert. This is not supported for all configurations of models and can yield errors.
You are using a model of type lir-dpr to instantiate a model of type bert. This is not supported for all configurations of models and can yield errors.
You are using a model of type lir-dpr to instantiate a model of type bert. This is not supported for all configurations of models and can yield errors.
The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'BertTokenizer'. 
The class this function is called from is 'BiEncoderBertTokenizerFast'.


     -> Applying Fusion (RRF)
 [4] Adding Cross-Encoder (webis/monoelectra-large)


You are using a model of type mono to instantiate a model of type electra. This is not supported for all configurations of models and can yield errors.
You are using a model of type mono to instantiate a model of type electra. This is not supported for all configurations of models and can yield errors.
You are using a model of type mono to instantiate a model of type electra. This is not supported for all configurations of models and can yield errors.
You are using a model of type mono to instantiate a model of type electra. This is not supported for all configurations of models and can yield errors.
The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'ElectraTokenizer'. 
The class this function is called from is 'CrossEncoderElectraTokenizerFast'.



Running Pipeline: BM25+RM3+BiEnc+RRF+CE
calling sliding on df of 5000 rows


passaging:   0%|          | 0/5000 [00:00<?, ?it/s]

Pipeline finished in 0:04:09.751001


pt.Experiment:   0%|          | 0/1 [00:00<?, ?system/s]

                    name        RR  P@10   nDCG@10  nDCG@100
0  BM25+RM3+BiEnc+RRF+CE  0.591659  0.39  0.264471  0.302612


,name,RR,P@10,nDCG@10,nDCG@100
0,BM25+RM3+BiEnc+RRF+CE,0.591659,0.39,0.264471,0.302612


In [33]:
run_toggled_experiment(
    dataset,
    SPARSE_PATH,
    dense_path,
    topics_variant='text',
        
    # --- TOGGLES ---
    use_bm25f=False,        # Strong title baseline
    use_rm3=False,          # Essential for short queries
    use_bi_encoder=True,    # Optional: Turn on for deep semantic search
    use_rrf=False,           # Optional: Only works if bi_encoder is True
    use_cross_encoder=False,# The precision booster
        
    # --- PARAMETERS ---
    bi_model_id=BI_MODEL,
    window_len=300,        # 300 tokens is robust for web docs
    window_stride=150,     # 50% overlap
    bi_top_k=1000,         # Rerank depth for Bi-Encoder
    ce_top_k=100,          # Rerank depth for Cross-Encoder
)

 [1] Using Standard BM25
 [3] Adding Bi-Encoder (webis/bert-bi-encoder)


You are using a model of type lir-dpr to instantiate a model of type bert. This is not supported for all configurations of models and can yield errors.
You are using a model of type lir-dpr to instantiate a model of type bert. This is not supported for all configurations of models and can yield errors.
You are using a model of type lir-dpr to instantiate a model of type bert. This is not supported for all configurations of models and can yield errors.
You are using a model of type lir-dpr to instantiate a model of type bert. This is not supported for all configurations of models and can yield errors.
The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'BertTokenizer'. 
The class this function is called from is 'BiEncoderBertTokenizerFast'.



Running Pipeline: BiEnc
Pipeline finished in 0:00:00.135730


pt.Experiment:   0%|          | 0/1 [00:00<?, ?system/s]

    name        RR   P@10   nDCG@10  nDCG@100
0  BiEnc  0.523022  0.314  0.229803  0.250048


,name,RR,P@10,nDCG@10,nDCG@100
0,BiEnc,0.523022,0.314,0.229803,0.250048


From these experiments we can see that:
- BM25+CE with parameters window=300, stride=150, ce_model="webis/monoelectra-large" performs the best for the given dataset
- Query Expansion via RM3 doesn't aid at all as it may be adding irrelevant terms
- Using Bi-enocder as a parallel retrieval and combining with BM25 also is not fruitful

#### Let's trying fusing a model of BM25+CE and Bi_encoder

In [66]:
def rrf_fuse_files(file_path_1, file_path_2, k=60):
    print(f"Fusing:\n A: {file_path_1}\n B: {file_path_2}")
    
    res_1 = pt.io.read_results(file_path_1)
    res_2 = pt.io.read_results(file_path_2)
    
    # 2. Add RRF Scores
    # Score = 1 / (k + rank)
    res_1['rrf_score'] = 1 / (k + res_1['rank'])
    res_2['rrf_score'] = 1 / (k + res_2['rank'])
    
    # We merge the two dataframes
    combined = pd.concat([res_1[['qid', 'docno', 'rrf_score']], 
                          res_2[['qid', 'docno', 'rrf_score']]])
    
    # Sum scores for the same (qid, docno) pair
    fused = combined.groupby(['qid', 'docno'])['rrf_score'].sum().reset_index()
    
    fused = fused.rename(columns={'rrf_score': 'score'})
    # Sort by score descending
    fused = fused.sort_values(['qid', 'score'], ascending=[True, False])
    # Recalculate rank
    fused['rank'] = fused.groupby('qid').cumcount()
    
    return fused

# Ensemble with BM25+CE and BI_ENC
file_bm25_ce = "run_20260109_0010_BM25_CE.res"
file_bi_enc  = "run_20260109_0311_BiEnc.res"

In [46]:
# Fuse them!
fused_run = rrf_fuse_files(file_bm25_ce, file_bi_enc, k=5)

# 4. Evaluate the Ensemble
metrics = [nDCG@10, nDCG@100, P@10, RR(rel=1)]
pt.Experiment(
    [fused_run],
    dataset.get_topics(variant='text'),
    dataset.get_qrels(),
    eval_metrics=metrics,
    names=["Ensemble"]
)

Fusing:
 A: run_20260109_0010_BM25_CE.res
 B: run_20260109_0311_BiEnc.res


,name,RR,P@10,nDCG@10,nDCG@100
0,Ensemble,0.708953,0.454,0.341521,0.374339


Since our Bi_Encoder is bad than BM25+CE, it doesn't help in fusing them. 

### LLM Query Expansion

In [8]:
import pandas as pd
from openai import OpenAI
from tqdm import tqdm
from secret import GWDG_API_KEY

In [9]:
def generate_hyde_gwdg(
    dataset, 
    api_key, 
    variant='text', 
    model_name="meta-llama-3.1-70b-instruct",
    system_prompt="You are a helpful assistant. Write a concise, factual passage that answers the user's question directly.",
    prompt_template="Question: {query}\nPassage:",
    temperature=0.7,
    max_tokens=150
):
    print(f"--- Connecting to GWDG Chat AI ({model_name}) ---")
    client = OpenAI(
        base_url="https://chat-ai.academiccloud.de/v1", 
        api_key=api_key
    )
    
    topics = dataset.get_topics(variant=variant).copy()
    new_queries = []
    
    print(f"--- Generating Hypothetical Documents (Temp={temperature}) ---")
    for q in tqdm(topics['query'], desc="Hallucinating"):
        try:
            # Format the user message dynamically
            user_content = prompt_template.format(query=q)
            
            response = client.chat.completions.create(
                model=model_name,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_content}
                ],
                temperature=temperature,
                max_tokens=max_tokens
            )
            
            fake_doc = response.choices[0].message.content.strip()
            fake_doc = fake_doc.replace("\n", " ") # Flatten for BM25
            
            # Combine: Query + Hallucination
            combined_query = f"{q} {fake_doc}"
            new_queries.append(combined_query)
            
        except Exception as e:
            print(f"Error on query '{q}': {e}")
            new_queries.append(q) # Fallback

    topics['query_original'] = topics['query']
    topics['query'] = new_queries
    print(topics['query_original'])
    print(topics['query'])
    
    return topics

In [10]:
def run_hyde_experiment(dataset, custom_topics, sparse_path, ce_model_id, batch_size=32):
    # 1. Setup BM25 (The Retriever)
    index = pt.IndexFactory.of(sparse_path)
    # We rely on BM25 to match the "fake answer" to "real documents"
    retriever = pt.terrier.Retriever(index, wmodel="BM25")

    # 2. Setup Cross-Encoder (The Filter)
    ce = CrossEncoder(
        model_name_or_path=ce_model_id,
        text_field="text",
        batch_size=max(1, batch_size // 2), 
        use_gpu=True
    )
    if hasattr(ce, "_module"): ce._module.to("cuda")

    # 3. Build Pipeline
    # Note: We do NOT use RM3 here because the LLM already expanded the query
    pipeline = (
        (retriever % 100) # Retrieve top 100 using HyDE query
        >> pt.text.get_text(dataset, "text") 
        >> pt.text.sliding(length=300, stride=150, text_attr='text', prepend_attr=None)
        >> ce 
        >> pt.text.max_passage()
    )
    
    name = "BM25_HyDE_CE_Large"
    print(f"\nRunning Pipeline: {name}")

    # 4. EXECUTE with CUSTOM TOPICS
    start_time = datetime.datetime.now()
    run_out = pipeline.transform(custom_topics)
    print(f"Pipeline finished in {datetime.datetime.now() - start_time}")

    # 5. Evaluate
    metrics = [nDCG@10, nDCG@100, P@10, RR(rel=1)]
    exp = pt.Experiment(
        [run_out],
        custom_topics,        # Pass custom topics here too for ID matching
        dataset.get_qrels(),
        eval_metrics=metrics,
        names=[name],
        verbose=True,
        validate=False 
    )
    
    timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M")
    pt.io.write_results(run_out, f"run_{timestamp}_{name}.res")
    print(exp)
    return exp

In [14]:
SPARSE_PATH = "./indices/sparse_clueweb"
DENSE_ROOT  = "./indices/dense_clueweb"
BI_MODEL = "webis/bert-bi-encoder" 
CE_MODEL = "webis/monoelectra-large"
    
dense_path = build_indexes(dataset, SPARSE_PATH, DENSE_ROOT, BI_MODEL)

hyde_topics_keywords = generate_hyde_gwdg(
    dataset, 
    api_key=GWDG_API_KEY,
    model_name="meta-llama-3.1-70b-instruct",
    system_prompt="You are a helpful assistant. Write a concise, factual passage that answers the user's question directly.",
    prompt_template="Question: {query}\nPassage:",
    temperature=0.7,
    max_tokens=150
)

# Run the pipeline
run_hyde_experiment(
    dataset, 
    hyde_topics_keywords,             
    SPARSE_PATH, 
    "webis/monoelectra-large"
)

--- Connecting to GWDG Chat AI (meta-llama-3.1-70b-instruct) ---
--- Generating Hypothetical Documents (Temp=0.7) ---


Hallucinating: 100%|██████████████████████████████████████| 50/50 [03:13<00:00,  3.87s/it]


0                              identifying spider bites
1                               history of orcas island
2                                         tooth abscess
3                                   barrett's esophagus
4                                           teddy bears
5                        patron saint of mental illness
6                                 holes by louis sachar
7                                              hip roof
8                                         carpenter bee
9                            the american revolutionary
10                            folk remedies sore throat
11                                         balding cure
12                               evidence for evolution
13                     tribe formerly living in alabama
14                                           F5 tornado
15                             symptoms of heart attack
16                                 feliz navidad lyrics
17                                  benefits of 

You are using a model of type mono to instantiate a model of type electra. This is not supported for all configurations of models and can yield errors.
You are using a model of type mono to instantiate a model of type electra. This is not supported for all configurations of models and can yield errors.
You are using a model of type mono to instantiate a model of type electra. This is not supported for all configurations of models and can yield errors.
You are using a model of type mono to instantiate a model of type electra. This is not supported for all configurations of models and can yield errors.
The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'ElectraTokenizer'. 
The class this function is called from is 'CrossEncoderElectraTokenizerFast'.



Running Pipeline: BM25_HyDE_CE_Large
calling sliding on df of 5000 rows


passaging:   0%|          | 0/5000 [00:00<?, ?it/s]

Pipeline finished in 0:12:20.494688


pt.Experiment:   0%|          | 0/1 [00:00<?, ?system/s]

                 name        RR   P@10   nDCG@10  nDCG@100
0  BM25_HyDE_CE_Large  0.596172  0.362  0.254356  0.290068


,name,RR,P@10,nDCG@10,nDCG@100
0,BM25_HyDE_CE_Large,0.596172,0.362,0.254356,0.290068


In [11]:
hyde_topics_keywords = generate_hyde_gwdg(
    dataset, 
    api_key=GWDG_API_KEY,
    model_name="meta-llama-3.1-70b-instruct",
    system_prompt="You are an expert search engine optimizer. Directly generate 20 highly relevant keywords and synonyms for the search query.",
    prompt_template="Question: {query}\nPassage:",
    temperature=0.4,
    max_tokens=100
)

# can run the pipeline before, but moved it to the next cell 
# since we have limited api calls with the GWDG_API_KEY and we set the topics already

--- Connecting to GWDG Chat AI (meta-llama-3.1-70b-instruct) ---
--- Generating Hypothetical Documents (Temp=0.4) ---


Hallucinating: 100%|██████████████████████████████████████| 50/50 [02:36<00:00,  3.13s/it]


0                              identifying spider bites
1                               history of orcas island
2                                         tooth abscess
3                                   barrett's esophagus
4                                           teddy bears
5                        patron saint of mental illness
6                                 holes by louis sachar
7                                              hip roof
8                                         carpenter bee
9                            the american revolutionary
10                            folk remedies sore throat
11                                         balding cure
12                               evidence for evolution
13                     tribe formerly living in alabama
14                                           F5 tornado
15                             symptoms of heart attack
16                                 feliz navidad lyrics
17                                  benefits of 

NameError: name 'SPARSE_PATH' is not defined

In [13]:
SPARSE_PATH = "./indices/sparse_clueweb"
DENSE_ROOT  = "./indices/dense_clueweb"
BI_MODEL = "webis/bert-bi-encoder" 
CE_MODEL = "webis/monoelectra-large"
    
dense_path = build_indexes(dataset, SPARSE_PATH, DENSE_ROOT, BI_MODEL)

# Run the pipeline
run_hyde_experiment(
    dataset, 
    hyde_topics_keywords,             
    SPARSE_PATH, 
    "webis/monoelectra-large"
)

You are using a model of type mono to instantiate a model of type electra. This is not supported for all configurations of models and can yield errors.
You are using a model of type mono to instantiate a model of type electra. This is not supported for all configurations of models and can yield errors.
You are using a model of type mono to instantiate a model of type electra. This is not supported for all configurations of models and can yield errors.
You are using a model of type mono to instantiate a model of type electra. This is not supported for all configurations of models and can yield errors.
The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'ElectraTokenizer'. 
The class this function is called from is 'CrossEncoderElectraTokenizerFast'.



Running Pipeline: BM25_HyDE_CE_Large
calling sliding on df of 5000 rows


passaging:   0%|          | 0/5000 [00:00<?, ?it/s]

Pipeline finished in 0:11:35.727008


pt.Experiment:   0%|          | 0/1 [00:00<?, ?system/s]

                 name        RR   P@10   nDCG@10  nDCG@100
0  BM25_HyDE_CE_Large  0.534964  0.372  0.261261  0.292594


,name,RR,P@10,nDCG@10,nDCG@100
0,BM25_HyDE_CE_Large,0.534964,0.372,0.261261,0.292594


### Energy

In [12]:
import time
import pandas as pd
import pyterrier as pt
from codecarbon import EmissionsTracker

bm25 = pt.terrier.Retriever(index, wmodel="BM25")
ce = CrossEncoder(
    model_name_or_path="webis/monoelectra-large",
    text_field="text",
    batch_size=32, 
    use_gpu=True
)
ce._module.to(device)
ce._module.device

# Reconstruct the Full Pipeline
# BM25 (Top 100) -> Fetch Text -> Sliding Window -> CrossEncoder -> Max Passage
pipeline = (
    (bm25 % 100)                                      
    >> pt.text.get_text(dataset, "text")              
    >> pt.text.sliding(length=300, stride=150, text_attr='text', prepend_attr=None) 
    >> ce                                             
    >> pt.text.max_passage()                          
)

# Measurement Function
def measure_efficiency(pipeline, topics, run_name="BM25+CE_Large"):
    print(f"--- Measuring Efficiency: {run_name} ---")
    
    tracker = EmissionsTracker(
        project_name=run_name, 
        output_dir="./", 
        measure_power_secs=1,
        log_level='error',
        save_to_file=True 
    )
    
    tracker.start()
    
    start_time = time.time()
    
    res = pipeline.transform(topics) 
    
    end_time = time.time()
    emissions = tracker.stop()
    
    total_seconds = end_time - start_time
    latency_ms = (total_seconds / len(topics)) * 1000  # ms per query
    
    print(f"   Latency:    {latency_ms:.2f} ms/query")
    print(f"   Emissions:  {emissions:.6f} kgCO2")
    
    return {"Model": run_name, "Latency": latency_ms, "Emissions": emissions}

# Run Measurement
stats = measure_efficiency(pipeline, dataset.get_topics(variant='text'))

# # Run on just 10 queries to get the numbers instantly
# stats = measure_efficiency(pipeline, dataset.get_topics(variant='text').head(10))
pd.DataFrame([stats])

You are using a model of type mono to instantiate a model of type electra. This is not supported for all configurations of models and can yield errors.
You are using a model of type mono to instantiate a model of type electra. This is not supported for all configurations of models and can yield errors.
You are using a model of type mono to instantiate a model of type electra. This is not supported for all configurations of models and can yield errors.
You are using a model of type mono to instantiate a model of type electra. This is not supported for all configurations of models and can yield errors.
The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'ElectraTokenizer'. 
The class this function is called from is 'CrossEncoderElectraTokenizerFast'.


--- Measuring Efficiency: BM25+CE_Large ---
calling sliding on df of 5000 rows


passaging:   0%|          | 0/5000 [00:00<?, ?it/s]

   Latency:    4724.35 ms/query
   Emissions:  0.013723 kgCO2


,Model,Latency,Emissions
0,BM25+CE_Large,4724.353781,0.013723


In [13]:
print(pd.DataFrame([stats]))

           Model      Latency  Emissions
0  BM25+CE_Large  4724.353781   0.013723


## Discussion

Use the following marked cells to respond to the following prompts

### 1. Describe your retrieval model and why you implemented it in the way you did.

(250 words max)



The final model is a Retrieve and Rerank pipeline consisting of BM25 (Sparse Retrieval) followed by a Cross-Encoder (MonoElectra-Large) reranker.

This specific architecture was implemented to navigate the "messy" nature of the ClueWeb dataset:

- First Stage: BM25 is prioritized over dense retrieval because our experiments revealed a severe domain shift. The `webis/bert-bi-encoder` which is trained on clean questions, failed to handle the noisy web forums of the corpus, often filtering out relevant documents that BM25 successfully caught. Consequently, BM25 proved more robust than either a pure Bi-Encoder or a hybrid (BM25 + Bi-Encoder) approach.
- Second Stage: We utilized `webis/monoelectra-large` with a reranking depth of 100. Switching from "base" to "large" yielded a +3.6% boost in RR, suggesting that model capacity was a primary driver of effectiveness for this task.
- Context Management: We implemented a sliding window of 300 tokens with a stride of 150. Experiments identified this as the "sweet spot": shorter windows (150) cut off necessary context, while longer ones (450) introduced too much noise from footers and navigation bars.

Furthermore, Query Expansion via RM3 failed to improve performance, likely by introducing irrelevant terms. Attempting expansion via the `meta-llama-3.1-70b-instruct` LLM also degraded performance. This failure was likely caused by the LLM prefixing the response with conversational filler (e.g., "Here are 20 highly relevant keywords..."), which was then truncated at 150 tokens, effectively injecting garbage data into the search query.

### 2. Describe how you could improve your model further and any limitations of your retrieval model.

(250 words max)

Limitations:

- Extreme Latency: As measured, the model requires 4,724 ms (4.7s) per query. This makes real-time deployment impossible, as the Cross-Encoder must perform heavy self-attention on 100 documents for every single search.
- Recall Ceiling: The pipeline is strictly bound by the recall of BM25. If a document lacks exact keyword overlap, the Cross-Encoder never sees it.
- Query Expansion Failure: The attempt to fix the recall issue with LLM-based Query Expansion failed (scoring 0.59 vs baseline). This was due to the injection of conversational filler words (as described above) and possibly "hallucination drift," where the LLM's vocabulary did not match the noisy corpus.


Improvements:

- Preprocess Query Expansion: Strip redundant conversational phrases (e.g., "Here are the keywords") from the LLM output to isolate only the effective search terms.
- Stricter System Prompt: Refine the prompt to enforce direct keyword output without conversational filler which will eliminating the need for post-processing.
- Domain Adaptation: Fine-tune the query generator to align hallucinations with the specific ClueWeb domain, reducing the vocabulary mismatch we observed.
- Knowledge Distillation: Distill the Cross-Encoder into a smaller architecture like a Bi-Encoder or Late Interaction model (ColBERT). Training on the teacher's scores would fix the domain shift issue while reducing latency from 4.7s to <100ms via pre-computation.


### 3. Describe the effectiveness--efficiency trade-offs made by your retrieval model (storage/computation/energy/etc).

see lecture on _Retrieval Efficiency_

(250 words max)

The final model explicitly prioritizes Effectiveness (nDCG@10: 0.375) at a massive cost to Efficiency, placing it deep in the "Red" zone of Green IR evaluation.

- Computation & Latency: Unlike sparse lookups which take milliseconds, the Cross-Encoder performs expensive BERT self attention ($O(N^2)$) on the top 100 documents. The measurements show a staggering latency of 4,724 ms (4.7s) per query, which is approximately 500x slower than a standard BM25 system (<10ms).
- Energy Consumption: By selecting the "Large" architecture, we significantly increased the floating point operations (FLOPs). The EmissionTracker calculated 0.0137 kgCO₂ emissions for our small batch alone.
    - Context: 0.0137 kg (13.7 grams) of $CO_2$ is roughly equivalent to fully charging a smartphone twice. Generating this much carbon for just a handful of search queries highlights why this architecture is unsustainable for a real world search engine handling millions of users.
- Storage: Conversely, our storage footprint is efficient. By rejecting dense retrieval indices (which require large RAM for vectors), the model relies only on the compact inverted index.

In summary, the model achieved high retrieval quality (0.375 nDCG@10) but created a system that is too slow and energy intensive for practical and large-scale use.